# Calculate long-term statistics over a polygon

In this notebook, we use VirtualiZarr to initialize a zarr-like virtual dataset that enables efficient indexing across multiple netCDF files at once.

We will use this data to calculate precipitation statistics on a region defined by a HUC-8 watershed boundary.

In [1]:
import s3fs
import xarray as xr
import json
import fsspec
import requests
import rasterio
import numpy as np
from tqdm.notebook import tqdm
from shapely.geometry import Point,Polygon
#from virtualizarr import open_virtual_dataset
from pathlib import Path
from datetime import datetime,timedelta
from multiprocessing import Pool

## Specify watershed and analysis time period

`lat` and `lon` must be land points in the NLDAS-3 domain,
and the start and end time should be within the supported 2001-2024 timeline.

In [2]:
## inclusive start and end days for annual long term stats.
start_month_day = (8,1)
end_month_day = (10,31)
## years to include in long term stats.
analysis_years = list(range(2010, 2020))
extract_feats = ["Rainf", "Tair_min", "Tair_max", "Qair"]
nworkers = 2

## point inside watershed to extract
lat,lon = 35,-110

## Extract full-domain latitude and longitude

Use the static parameter file to load the latitude and longitude arrays

In [3]:
## Establish a connection with the s3 bucket
s3 = s3fs.S3FileSystem(anon=True)

## s3 bucket location of static file
static_file = "s3://nasa-waterinsight/NLDAS3/static/NLDAS-3_dominant-soil-vegetation.nc"

## Open the file in a context manager to extract the coordinates and feature labels.
with s3.open(static_file) as ncfile:
    ds = xr.open_dataset(ncfile, engine="h5netcdf")
    full_dom_lats = ds["lat"].load().to_numpy()
    full_dom_lons = ds["lon"].load().to_numpy()

## Retrieve a HUC-8 polygon

Query the USGS Watershed Boundary Dataset (WBD) API for the
watershed surrounding the geographic point defined above

In [4]:
## USGS WBD endpoint; layer 4 corresponds to HUC-8 polygons
base_url = "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query"

## query string params for the request
params = {
    "geometry": f"{lon},{lat}",
    "geometryType": "esriGeometryPoint",
    "inSR": "4326", # expect wgs84 spatial reference
    "outSR": "4326", ## output wgs84 spatial reference
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "huc8,name", ## fields to return
    "returnGeometry": "true", ## include poly coordinates
    "f": "json" ## request json response format
    }

try:
    ## issue a GET request with the parameters
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    
    ## check if any features were returned
    if "features" in data and len(data["features"]) > 0:
        feature = data["features"][0]
        attributes = feature.get("attributes", {})
    else:
        print(f"No HUC-8 boundary found for point ({lat}, {lon}). Ensure the point is within the US.")
        feature = None
        attributes = None
except requests.exceptions.RequestException as e:
    print(f"An error occurred while querying the API: {e}")
    feature = None
    attributes = None

if feature is None or attributes is None:
    raise ValueError("Failed request")

print(f"Point ({lat}, {lon}) in HUC8-{attributes['huc8']}")
print(f"Subbasin Name: {attributes['name']}")

Point (35, -110) in HUC8-15020007
Subbasin Name: Lower Puerco


## Identify a domain subset mask for the selection

Restrict the full NLDAS-3 domain to a minimum sized rectangle containing the watershed,
then mask only points that fall within the polygon.

Since the full domain is so large, restricting the search size before rigorously determining
whether pixels are inside the polygon saves a lot of time.

Then, use shapely to develop a mask over pixels that are strictly within the polygon.

In [5]:
## get outer coords of the smallest rectangle completely containing the watershed
huc_poly = np.asarray(feature["geometry"]["rings"][0])
lon0,lat0 = np.amin(huc_poly, axis=0)
lonf,latf = np.amax(huc_poly, axis=0)

## get index slices for rectangular subdomain
lat_ixs_subgrid = np.where((full_dom_lats >= lat0) & (full_dom_lats <= latf))[0]
lon_ixs_subgrid = np.where((full_dom_lons >= lon0) & (full_dom_lons <= lonf))[0]
sub_lat_slice = slice(lat_ixs_subgrid[0], lat_ixs_subgrid[-1] + 1)
sub_lon_slice = slice(lon_ixs_subgrid[0], lon_ixs_subgrid[-1] + 1)

## declare a list of shapely points for each pixel in the 2d subdomain
sub_lats,sub_lons = np.meshgrid(
    full_dom_lats[sub_lat_slice],
    full_dom_lons[sub_lon_slice],
    indexing="ij",
    )
points = list(enumerate(map(Point, zip(sub_lons.ravel(), sub_lats.ravel()))))

## declare a shapely polygon for the retrieved watershed
poly = Polygon(huc_poly.tolist())

## get the 1d index of points within the polygon
valid_ixs_points = list(filter(
    lambda p:poly.contains(p[1]),
    points
    ))
sub_ixs_1d,_ = zip(*valid_ixs_points)

## make a 2d bool mask setting pixels in the rectangular subdomain
## AND the watershed polygon to True
sub_mask = np.full(sub_lats.size, False)
sub_mask[(sub_ixs_1d,)] = True
sub_mask = np.reshape(sub_mask, sub_lats.shape)

## Determine available file keys 

Query the s3 bucket for all forcing files within the time period.

In [6]:
## for each analysis year, build a list of forcing file keys for each date to be included.
all_keys = []
for y in analysis_years:
    d0 = datetime(y, *start_month_day)
    df = datetime(y, *end_month_day)

    ## if df<d0, the date wrange wraps years, so get dates outside of the range.
    if df < d0:
        year_start = datetime(y, 1, 1)
        year_end = datetime(y, 12, 31)
        dates_early = [
            year_start + timedelta(days=x)
            for x in range((d0 - year_start).days + 1)
            ]
        dates_late = [
            df + timedelta(days=x)
            for x in range((year_end - df).days + 1)
            ]
        cur_dates = dates_early + dates_late
    ## otherwise get dates inside the range
    else:
        cur_dates = [d0 + timedelta(days=x) for x in range((df - d0).days + 1)]

    ## list the valid months and restrict file keys to those in the time range
    daily_keys = [
        (datetime.strptime(Path(k).stem.split(".")[1], "A%Y%m%d"), f"s3://{k}")
        for m in list(set(d.month for d in cur_dates))
        for k in sorted(s3.ls(f"nasa-waterinsight/NLDAS3/forcing/daily/{y:04}{m:02}/"))
        ]
    
    ## retain outside or inside this year's interval depending on whether the date
    ## range wraps years.
    if df < d0:
        all_keys += [fk[1] for fk in daily_keys if fk[0] <= d0 and fk[0] >= df]
    else:
        all_keys += [fk[1] for fk in daily_keys if fk[0] >= d0 and fk[0] <= df]


## Accumulate stats across files

In [12]:
from time import perf_counter
def _get_watershed_stats(nc_key):
    t0 = perf_counter()
    with s3.open(nc_key) as ncfile:
        t1 = perf_counter()
        ds = xr.open_dataset(
            ncfile,
            engine="h5netcdf",
            decode_cf=False,
            decode_times=False,
            decode_coords=False,
            )
        t2 = perf_counter()
        sub = ds.isel({
            "lat":sub_lat_slice,
            "lon":sub_lon_slice,
            "time":0,
            })
        t3 = perf_counter()
        sub = np.stack([
            sub[k].to_numpy()
            for k in extract_feats
            ], axis=-1)
        t4 = perf_counter()
    return sub[sub_mask],(t0,t1,t2,t3,t4)

for k in tqdm(all_keys):
    tinit = perf_counter()
    s,times= _get_watershed_stats(k)
    print([f"{v:.04f}" for v in np.diff(times)])

  0%|          | 0/920 [00:00<?, ?it/s]

['0.0006', '3.6180', '0.0008', '1.6025']
['0.0006', '4.0725', '0.0007', '1.4018']
['0.0006', '4.2870', '0.0007', '1.7236']
['0.0006', '4.1019', '0.0007', '1.5816']
['0.0010', '4.0274', '0.0007', '1.7538']
['0.0006', '4.5724', '0.0007', '1.8991']
['0.0006', '3.8957', '0.0008', '1.6608']
['0.0007', '3.6209', '0.0007', '2.0817']
['0.0006', '4.5567', '0.0008', '1.4293']
['0.0006', '4.2571', '0.0007', '2.1659']
['0.0006', '4.1483', '0.0008', '1.8756']
['0.0007', '4.1698', '0.0008', '1.8429']
['0.0006', '4.2636', '0.0008', '1.5661']
['0.0006', '4.1172', '0.0013', '1.4429']
['0.0007', '4.5239', '0.0007', '2.2165']
['0.0007', '4.2920', '0.0008', '2.0607']
['0.0008', '4.2162', '0.0008', '1.5955']
['0.0006', '4.3565', '0.0007', '1.5334']
['0.0009', '4.0183', '0.0007', '1.7013']
['0.0006', '4.5620', '0.0007', '1.8236']
['0.0006', '4.1171', '0.0008', '1.8322']
['0.0006', '3.9898', '0.0007', '1.6620']
['0.0007', '4.2540', '0.0008', '1.8643']
['0.0007', '4.1269', '0.0008', '1.9643']
['0.0008', '4.37

Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3505, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_431/89557076.py", line 29, in <module>
    s,times= _get_watershed_stats(k)
  File "/tmp/ipykernel_431/89557076.py", line 20, in _get_watershed_stats
    sub = np.stack([
  File "/tmp/ipykernel_431/89557076.py", line 21, in <listcomp>
    sub[k].to_numpy()
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/xarray/core/dataarray.py", line 743, in to_numpy
    def nbytes(self) -> int:
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/xarray/core/variable.py", line 1245, in to_numpy
    keep_attrs = _get_keep_attrs(default=True)
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/xarray/core/variable.py", line 430, in data
    duck_array = self._data.get_duck_array()
  File "/srv/conda/envs/notebook/lib/python3.10/site-packages/

In [ ]:
# 1. List and sort files — ordering from key name determines time order
fs = s3fs.S3FileSystem(anon=True)  # or with credentials
keys = sorted(fs.ls("my-bucket/prefix/"))  # sorted() gives lexicographic time order
paths = [f"s3://{k}" for k in keys]

# 2. Open each file as a virtual dataset (reads only metadata/chunk refs, no array data)
virtual_ds_list = [
    open_virtual_dataset(
        path,
        indexes={},            # defer index loading; build combined index later
        reader_options={"storage_options": {"anon": True}},
        )
    for path in paths
    ]

# 3. Concatenate along time — this is pure metadata manipulation, zero data movement
combined = xr.concat(virtual_ds_list, dim="time")

# 4a. Write references as a Kerchunk JSON (portable, works with zarr v2 ecosystem)
refs = combined.virtualize.to_kerchunk(format="dict")
with open("combined.json", "w") as f:
    json.dump(refs, f)

# 4b. Or write as a Zarr v3 store directly (newer, preferred if your stack supports it)
combined.virtualize.to_zarr("combined.zarr")